# Lekcia 05 - Agentický RAG


## Nastavenie

Tento zápisník demonštruje vzor Agentic RAG (Retrieval-Augmented Generation) pomocou Microsoft Agent Framework.

**Predpoklady:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — váš koncový bod služby Azure AI Search
- `AZURE_SEARCH_API_KEY` — váš API kľúč služby Azure AI Search
- Deployment Azure OpenAI nakonfigurovaný prostredníctvom environmentálnych premenných
- Autentifikácia v Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Čo je Agentic RAG?

Tradičný RAG nasleduje pevný postup: vyhľadať dokumenty, potom vygenerovať odpoveď. **Agentic RAG** ide ďalej tým, že agentovi dáva autonómiu rozhodnúť, **kedy** a **ako** získať informácie.

S Agentic RAG môže agent:
- **Rozhodnúť sa**, či je potrebné vyhľadávanie pred odpoveďou na otázku
- **Vybrať** zdroj dát alebo nástroj na vyhľadávanie
- **Vyhodnotiť** získané výsledky a vykonať ďalšie vyhľadávanie, ak prvý pokus nestačí
- **Kombinovať** informácie z viacerých krokov vyhľadávania do zrozumiteľnej odpovede

To robí agenta flexibilnejším a presnejším v porovnaní so statickým postupom najprv vyhľadaj a potom vytvor odpoveď.


## Vytváranie vyhľadávacieho nástroja

V Agentic RAG sú externé zdroje údajov zabalené ako **nástroje**, ktoré môže agent vyvolať na požiadanie. Toto umožňuje agentovi považovať vyhľadávanie za len ďalšiu akciu, ktorú môže vykonať, namiesto povinného kroku.

Nižšie definujeme cestovateľskú databázu znalostí a sprístupníme ju ako nástroj, ktorý môže agent volať na vyhľadávanie informácií o destináciách.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Vytváranie RAG Agenta

Teraz vytvoríme agenta, ktorý je inštruovaný, aby **vždy získaval informácie pred odpovedaním**. Agent používa nástroj `search_travel_knowledge` na zakotvenie svojich odpovedí v databáze znalostí namiesto spoliehania sa na svoje vlastné tréningové dáta.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteratívne vyhľadávanie — Vzor tvorca-kontrolór

Kľúčovou výhodou Agentic RAG je **iteratívne vyhľadávanie**. Agent môže vykonať niekoľko kôl vyhľadávania na overenie, spresnenie alebo rozšírenie svojich počiatočných zistení — podobne ako pracovný proces „tvorca-kontrolór“:

1. **Krok tvorcu**: Agent získava počiatočné informácie a navrhuje odpoveď.
2. **Krok kontrolóra**: Agent vykonáva ďalšie vyhľadávania na overenie detailov alebo doplnenie medzier.

Nižšie je agentovi položená otázka, ktorá vyžaduje porovnanie viacerých destinácií, čo ho podnecuje k niekoľkým vyhľadávaniam.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Zhrnutie

V tejto lekcii ste sa naučili, ako vytvoriť systém **Agentic RAG** pomocou Microsoft Agent Framework:

- **Agentic RAG** umožňuje agentom autonómne rozhodovať, kedy načítať informácie, čím sa načítanie stáva dynamickým namiesto pevne daného.
- **Nástroje ako zdroje dát**: Externé znalostné bázy (ako Azure AI Search) sú zabalené ako nástroje, ktoré agent môže vyvolať.
- **Iteratívne načítanie**: vzor maker-checker umožňuje agentovi vykonať niekoľko kôl načítavania — hľadanie, overovanie a zdokonaľovanie — predtým, než vyprodukuje konečnú odpoveď.

V produkcii by ste nahradili in-memory `TRAVEL_KNOWLEDGE_BASE` skutočným indexom Azure AI Search na spracovanie veľkorozsahového vyhľadávania v cestovných dokumentoch.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zrieknutie sa zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatizované preklady môžu obsahovať chyby alebo nepresnosti. Originálny dokument v jeho pôvodnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie odporúčame profesionálny ľudský preklad. Nie sme zodpovední za akékoľvek nedorozumenia alebo nesprávne výklady vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
